In [ ]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from glob import glob

from monai.transforms import (
    Compose, LoadImageD, ScaleIntensityRangeD,
    RandSpatialCropSamplesD, EnsureTypeD, LambdaD
)
from monai.data import Dataset, DataLoader

import albumentations as A

from monai.transforms import MapTransform   
from PIL import Image


In [ ]:
class AlbumentationsD(MapTransform):
    def __init__(self, keys, aug):
        super().__init__(keys)
        self.aug = aug
    
    def __call__(self, data):
        d = dict(data)
        img = d[self.keys[0]]
        mask = d[self.keys[1]]

        img = np.asarray(img)
        mask = np.asarray(mask)

        if img.ndim == 3 and img.shape[0] in (1, 3) and img.shape[0] < img.shape[1]:
            img = np.transpose(img, (1, 2, 0))  # C,H,W -> H,W,C

        if mask.ndim == 3 and mask.shape[0] <= 3 and mask.shape[0] < mask.shape[1]:
            mask = np.transpose(mask, (1, 2, 0))  # C,H,W -> H,W,C

        augmented = self.aug(image=img, mask=mask)
        img_aug = augmented['image']
        mask_aug = augmented['mask']

        if img_aug.ndim == 3:
            img_aug = np.transpose(img_aug, (2, 0, 1))  # [C,H,W]

        if mask_aug.ndim == 3:
            mask_aug = np.transpose(mask_aug, (2, 0, 1))  # [C,H,W] RGB mask

        d[self.keys[0]] = img_aug.astype(np.float32)   # image
        d[self.keys[1]] = mask_aug.astype(np.uint8)    # label (RGB)

        return d

In [ ]:
def get_weather_transforms(
    rain=False,
    sunny=False,
    snow=False,
    foggy=False,
    clahe=False,
):
    Aug = []

    if clahe:
        Aug.append(
            A.CLAHE(clip_limit=4.0, tile_grid_size=(8,8), p=0.5)
        )

    Aug += [
        A.OneOf(
            [
                A.HueSaturationValue(
                    hue_shift_limit=0.1,
                    sat_shift_limit=0.1,
                    val_shift_limit=0.1,
                    p=0.7,
                ),
                A.RandomBrightnessContrast(
                    brightness_limit=0.15,
                    contrast_limit=0.15,
                    p=0.9,
                ),
            ],
            p=0.7,
        ),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Normalize(p=1.0),
    ]

    W = []
    if rain:
        W.append(
            A.RandomRain(
                slant_range=(-10, 10),
                drop_length=15,
                drop_width=1,
                drop_color=(200, 200, 200),
                blur_value=12,
                brightness_coefficient=0.52,
                rain_type='heavy',
                p=0.25,
            )
        )
    
    if sunny:
        W.append(
            A.RandomSunFlare(
                flare_roi=(0, 0, 1, 1),
                angle_range=(0, 1),
                num_flare_circles_range=(4, 8),
                src_radius=100,
                src_color=(255, 255, 255),
                p=0.25,
            )
        )

    if snow:
        W.append(
            A.RandomSnow(
                snow_point_range=(0.1, 0.3),
                brightness_coeff=2.5,
                p=0.25,
            )
        )

    if foggy:
        W.append(
            A.RandomFog(
                fog_coef_range=(0.3, 0.4),
                alpha_coef=0.1,
                p=0.01,
            )
        )

    return A.Compose(Aug + W)

In [ ]:
transform = Compose([
    LoadImageD(keys=["image", "label"]),
    AlbumentationsD(keys=["image", "label"],
                    aug=get_weather_transforms(
                        rain=False, sunny=False, snow=False, foggy=False, clahe=True
                    )),
    RandSpatialCropSamplesD(
        keys=["image", "label"],
        roi_size=[512, 512],
        num_samples=4,
        random_size=False,
    ),
    LambdaD(keys=['label'], func=rgb_to_cls),
    EnsureTypeD(keys=["image", "label"]),
])

In [ ]:
ROOT = Path(os.getcwd()).parent
DATA_ROOT = ROOT / "uav-project"/ "data" / "UAVid" / "uavid_train" 

images = sorted(glob(str(DATA_ROOT / "seq*" / "Images" / "*.png")))
labels = sorted(glob(str(DATA_ROOT / "seq*" / "Labels" / "*.png")))

data = [{"image": img, "label": lbl} for img, lbl in zip(images, labels)]

In [ ]:
UAVID_CLS = [
    'Background clutter', # 0
    'Building',           # 1
    'Road',               # 2
    'Tree',               # 3
    'Low vegetation',     # 4
    'Moving car',         # 5
    'Static car',         # 6
    'Human',              # 7
]

UAVID_COLORMAP = [
    (0, 0, 0),          # Background Clutter
    (128, 0, 0),        # Building
    (128, 64, 128),     # Road
    (0, 128, 0),        # Tree
    (128, 128, 0),      # Low vegetation
    (64, 0, 128),       # Moving car
    (192, 0, 192),      # Static car
    (64, 64, 0),        # Human 
]

UAVID_PASTE_RATIO = [
    0,     # Background Clutter
    0,     # Building
    0,     # Road
    0,     # Tree
    0,     # Low vegetation
    0.25,     # Moving car
    0.25,     # Static car
    0.5,     # Human 
]

COLOR2ID = {rgb: idx for idx, rgb in enumerate(UAVID_COLORMAP)}
ID2COLOR = {idx: rgb for idx, rgb in enumerate(UAVID_COLORMAP)}

def rgb_to_cls(mask):
    # Mask shape: (3, H, W)
    _, h, w = mask.shape
    mapped_mask = np.zeros((h, w), dtype=np.uint8)

    for rgb, cls_id in COLOR2ID.items():
        matches = np.all(mask == np.array(rgb)[:, None, None], axis=0)
        mapped_mask[matches] = cls_id

    return mapped_mask[np.newaxis, ...].astype(np.int64)  # add channel dimension
def convert_data(raw_data):
    processed_data = []

    for item in raw_data:
        # Load image
        img = item["image"]
        if isinstance(img, str):
            img = np.array(Image.open(img).convert("RGB"))
        else:
            img = np.asarray(img, dtype=np.uint8)  # ensure it's a numpy array

        # Load mask
        mask = item["label"]
        if isinstance(mask, str):
            mask = np.array(Image.open(mask).convert("RGB"))
        else:
            mask = np.asarray(mask, dtype=np.uint8)

        # Ensure correct shapes
        assert img.ndim == 3 and img.shape[2] == 3, f"Image must be H×W×3, got {img.shape}"
        assert mask.ndim == 3 and mask.shape[2] == 3, f"Mask must be H×W×3, got {mask.shape}"

        # Convert mask to channel-first (3,H,W)
        mask = mask.transpose(2, 0, 1)

        processed_data.append([img, mask])

    return processed_data

In [ ]:
class CAPAugmentation():
    def __init__(self, data, num_paste=100, paste_ratio=None, seed=42):
        """
        Cross-Image Copy-Paste Augmentation

        Args:
            data: list of dicts with 'image' and 'label'
            num_paste: number of objects to paste per image
            paste_ratio: list of ratios per class
            seed: random seed
        """
        random.seed(seed)
        np.random.seed(seed)

        self.data = convert_data(data)
        self.num_paste = num_paste
        if paste_ratio is None:
            self.paste_ratio = [0.0] * len(UAVID_CLS)  # default zero for all classes
        else:
            self.paste_ratio = paste_ratio
        self.object_pool = defaultdict(list)

    # -------------------------
    # COPY
    # -------------------------
    @staticmethod
    def rgb_to_cls(mask):
        """Convert RGB mask to class indices"""
        # Mask shape: (3, H, W)
        _, h, w = mask.shape
        mapped_mask = np.zeros((h, w), dtype=np.uint8)

        for rgb, cls_id in COLOR2ID.items():
            matches = np.all(mask == np.array(rgb)[:, None, None], axis=0)
            mapped_mask[matches] = cls_id

        return mapped_mask[np.newaxis, ...].astype(np.int64)  # add channel dimension

    @staticmethod
    def cls_to_rgb(cls_mask):
        """
        Convert a class index mask to an RGB mask.

        Args:
            cls_mask: (1, H, W) array of class indices (int)

        Returns:
            rgb_mask: (3, H, W) uint8 RGB array
        """
        _, H, W = cls_mask.shape
        rgb_mask = np.zeros((3, H, W), dtype=np.uint8)

        for cls_id, rgb in ID2COLOR.items():
            pixels = cls_mask[0] == cls_id
            for c in range(3):
                rgb_mask[c][pixels] = rgb[c]

        return rgb_mask

    def extract_obj(self, img, class_mask, class_enum, radius=200, threshold=10000):
        """Extract object from image within circular region"""
        bin_mask = (class_mask.squeeze() == class_enum).astype(np.uint8)
        ys, xs = np.where(bin_mask)
        if len(ys) == 0:
            return None

        # Random center
        idx = np.random.randint(len(ys))
        center_y, center_x = ys[idx], xs[idx]

        H, W = bin_mask.shape
        y_grid, x_grid = np.ogrid[:H, :W]
        circle_mask = (x_grid - center_x)**2 + (y_grid - center_y)**2 <= radius**2
        obj_mask_binary = bin_mask & circle_mask

        if np.sum(obj_mask_binary) < threshold:
            return None

        # Prepare object image with alpha
        obj_img = np.zeros((*img.shape[:2], 4), dtype=img.dtype)
        obj_img[:, :, :3] = img
        obj_img[..., 3] = obj_mask_binary.astype(np.uint8) * 255

        # Prepare RGBA mask
        obj_mask = np.zeros_like(class_mask)
        obj_mask[0][obj_mask_binary] = class_enum

        obj_mask_rgb = self.cls_to_rgb(obj_mask)  # shape (3, H, W)
        obj_mask_rgb = np.transpose(obj_mask_rgb, (1, 2, 0))  # shape (H, W, 3)

        obj_mask_rgba = np.zeros((*obj_mask_rgb.shape[:2], 4), dtype=obj_mask_rgb.dtype)
        obj_mask_rgba[:, :, :3] = obj_mask_rgb
        obj_mask_rgba[..., 3] = obj_mask_binary.astype(np.uint8) * 255

        return [obj_img, obj_mask_rgba]

    def copy(self, img, mask):
        """Extract objects from image and add to pool"""
        # Convert to class mask (contains value 0-7)
        class_mask = self.rgb_to_cls(mask)
        # extract each class one time and add to obj pool
        for i, cls in enumerate(UAVID_CLS):
            # Limit to copy car and human class only
            if i not in [5, 6, 7]:
                continue
            obj = self.extract_obj(img, class_mask, i)
            if obj is not None:
                print(f'Added object {cls}')
                self.object_pool[cls].append(obj)

    def build_object_pool(self, samples=50, instance_per_cls=10):
        """
        Build object pool from samples.
        Extracts objects from random samples and limits pool size.
        """
        # Build pool
        copy_data = random.sample(self.data, min(samples, len(self.data)))
        for copy_img, copy_mask in copy_data:
            self.copy(copy_img, copy_mask)

        # Limit pool
        for cls in UAVID_CLS:
            objs = self.object_pool[cls]
            if len(objs) > instance_per_cls:
                self.object_pool[cls] = random.sample(objs, instance_per_cls)

        print("Pool built completed")

    # -------------------------
    # PASTE
    # -------------------------
    def paste(self, img: np.ndarray, mask: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        """Paste objects from pool onto image and mask"""
        img_out = img.copy().astype(np.float32)
        mask_out = mask.copy()

        for i, cls in enumerate(UAVID_CLS):
            num_to_paste = int(self.num_paste * self.paste_ratio[i])
            if num_to_paste == 0 or len(self.object_pool[cls]) == 0:
                continue

            for _ in range(num_to_paste):
                obj_img, obj_mask = random.choice(self.object_pool[cls])

                # obj_img shape: (H, W, 4) RGBA
                # obj_mask shape: (H, W, 4) RGBA

                # Get alpha channel from object image
                obj_alpha = obj_img[:, :, 3:4] / 255.0  # (H, W, 1)

                # Randomly place object in image
                h_obj, w_obj = obj_img.shape[:2]
                h_img, w_img = img_out.shape[:2]

                y_start = np.random.randint(0, max(1, h_img - h_obj + 1))
                x_start = np.random.randint(0, max(1, w_img - w_obj + 1))
                y_end = min(y_start + h_obj, h_img)
                x_end = min(x_start + w_obj, w_img)

                # Adjust object slicing if it goes out of bounds
                obj_y_start = 0
                obj_x_start = 0
                if y_start + h_obj > h_img:
                    h_obj = h_img - y_start
                if x_start + w_obj > w_img:
                    w_obj = w_img - x_start

                # Extract object region
                obj_img_region = obj_img[obj_y_start:obj_y_start+h_obj, obj_x_start:obj_x_start+w_obj]  # (h, w, 4)
                obj_mask_region = obj_mask[obj_y_start:obj_y_start+h_obj, obj_x_start:obj_x_start+w_obj]  # (h, w, 4)

                # Get alpha from object image
                obj_alpha_region = obj_img_region[:, :, 3:4] / 255.0  # (h, w, 1)

                # Alpha blend for image region
                img_region = img_out[y_start:y_end, x_start:x_end]  # (h, w, 3)
                img_blended = obj_img_region[:, :, :3] * obj_alpha_region + img_region * (1 - obj_alpha_region)
                img_out[y_start:y_end, x_start:x_end] = img_blended

                # Alpha blend for mask region (HWC format)
                obj_mask_rgb = obj_mask_region[:, :, :3]  # (h, w, 3)
                mask_region = mask_out[:, y_start:y_end, x_start:x_end].transpose(1, 2, 0)  # (h, w, 3)
                mask_blended = obj_mask_rgb * obj_alpha_region[:, :, 0:1] + mask_region * (1 - obj_alpha_region[:, :, 0:1])
                mask_out[:, y_start:y_end, x_start:x_end] = mask_blended.transpose(2, 0, 1).astype(np.uint8)

        return img_out.astype(np.uint8), mask_out

    # -------------------------
    # AUGMENT DATASET
    # -------------------------
    def augment_dataset(self, data_in=None):
        """
        Augment dataset by pasting objects onto images.
        
        Args:
            data_in: optional list of [img, mask] pairs to augment
            
        Returns:
            aug_data: list of dicts with 'image' and 'label'
        """
        data = data_in if data_in is not None else self.data
        aug_data = []
        for img, lbl in data:
            aug_img, aug_mask = self.paste(img, lbl)
            aug_data.append({'image': aug_img, 'label': aug_mask})
        return aug_data

In [ ]:
cap = CAPAugmentation(data[:10],paste_ratio=UAVID_PASTE_RATIO)
cap.build_object_pool()

In [ ]:
aug_data=cap.augment_dataset(data[20:28])

In [ ]:
def chw_to_hwc(x):
    """Convert CHW -> HWC"""
    return np.transpose(x, (1, 2, 0))
def normalize_0_1(arr):
    """Normalize array to [0,1]"""
    arr = arr.astype(np.float32)
    mn, mx = arr.min(), arr.max()
    if mx > mn:
        return (arr - mn) / (mx - mn)
    return arr
def display_augmented_data(aug_data, cols=4):
    """
    Display augmented images and masks.
    Args:
        aug_data: list of dicts [{'image': img, 'label': mask}, ...]
        cols: number of columns in grid
    """
    num_samples = len(aug_data)
    rows = int(np.ceil(num_samples / cols))
    rows *= 2  # double rows to show labels below images
    plt.figure(figsize=(4 * cols, 3 * rows))
    for i, sample in enumerate(aug_data):
        img = sample["image"]
        mask = sample["label"]
        # Image
        img_norm = normalize_0_1(img)
        plt.subplot(rows, cols, (i // cols) * 2 * cols + (i % cols) + 1)
        plt.imshow(img_norm)
        plt.axis("off")
        plt.title(f"Image {i}")
        # Mask
        if mask.shape[0] == 3:  # RGB mask
            mask_disp = chw_to_hwc(mask)
        else:  # single channel
            mask_disp = mask[0]
        plt.subplot(rows, cols, (i // cols) * 2 * cols + cols + (i % cols) + 1)
        plt.imshow(mask_disp)
        plt.axis("off")
        plt.title(f"Label {i}")
    plt.tight_layout()
    plt.show()
# Example usage:
# aug_sample = cap.augment_dataset(convert_data(sampled_data))
display_augmented_data(aug_sample)